# 02. Benchmark Selection

전처리 통합 데이터에서 국제 원유/제품 가격 후보를 만들고, 정유사 세전 공급가격을 기준으로 유종별 benchmark를 선택합니다.

이 노트북은 Google Colab에서 단독 실행되도록 구성되어 있습니다. 첫 설정 셀의 경로만 본인 Google Drive 구조에 맞게 수정한 뒤 위에서 아래로 실행하면 됩니다.


In [ ]:
# ============================================================
# 02 공통 경로 설정
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_PATH = os.path.join(ROOT_PATH, "data") + "/"
PROCESSED_PATH = os.path.join(ROOT_PATH, "preprocessed_data") + "/"

print(f"ROOT_PATH      = {ROOT_PATH}")
print(f"DATA_PATH      = {DATA_PATH}")
print(f"PROCESSED_PATH = {PROCESSED_PATH}")
BENCHMARK_OUTPUT_PATH = os.path.join(ROOT_PATH, "benchmark_selection") + "/"
print(f"BENCHMARK_OUTPUT_PATH = {BENCHMARK_OUTPUT_PATH}")

# 필요한 패키지 확인/설치
import importlib.util
import subprocess
import sys
import json

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "statsmodels": "statsmodels",
    "tqdm": "tqdm",
    "IPython": "ipython",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"[설치] 필요한 패키지 설치: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("[확인] 필요한 패키지가 이미 설치되어 있습니다.")


In [ ]:
# =========================
# 1) 패키지 import
# =========================
import numpy as np
import pandas as pd
import statsmodels.api as sm
from tqdm.auto import tqdm
from statsmodels.stats.diagnostic import acorr_ljungbox
from IPython.display import display


In [ ]:
# =========================
# 2) benchmark screening 함수
# =========================
def _to_datetime_index(df, date_col):
    out = df.copy()
    if date_col in out.columns:
        out[date_col] = pd.to_datetime(out[date_col])
        out = out.sort_values(date_col).set_index(date_col)
    else:
        out.index = pd.to_datetime(out.index)
        out = out.sort_index()
    return out

def _resample_w_sat_mean(s):
    s = pd.Series(s).dropna().copy()
    s.index = pd.to_datetime(s.index)
    return s.sort_index().resample("W-SAT").mean()

def _logdiff_pct(s):
    s = pd.Series(s).astype(float)
    return 100.0 * np.log(s).diff()

def _month_dummies(idx):
    idx = pd.to_datetime(idx)
    out = pd.get_dummies(pd.Index(idx.month), prefix="m", drop_first=True, dtype=float)
    out.index = idx
    return out

def _build_design(y_level, x_level, p_max, q_max, fixed_df=None):
    df = pd.DataFrame({
        "y_level": y_level.astype(float),
        "x_level": x_level.astype(float),
    }).dropna().copy()

    df["dy"] = _logdiff_pct(df["y_level"])
    df["dx"] = _logdiff_pct(df["x_level"])

    for i in range(1, p_max + 1):
        df[f"dy_lag_{i}"] = df["dy"].shift(i)

    for j in range(1, q_max + 1):
        df[f"dx_lag_{j}"] = df["dx"].shift(j)

    if fixed_df is not None and len(fixed_df.columns) > 0:
        fx = fixed_df.copy()
        fx.index = pd.to_datetime(fx.index)
        df = df.join(fx, how="left")

    return df.dropna().copy()

def _feature_cols(p, q, fixed_cols=None):
    fixed_cols = list(fixed_cols or [])
    return (
        [f"dy_lag_{i}" for i in range(1, p + 1)]
        + [f"dx_lag_{j}" for j in range(1, q + 1)]
        + fixed_cols
    )

def _fit_plain_ols(design_df, p, q, fixed_cols=None):
    cols = _feature_cols(p, q, fixed_cols=fixed_cols)
    y = design_df["dy"].astype(float)
    X = sm.add_constant(design_df[cols].astype(float), has_constant="add")
    return sm.OLS(y, X).fit()

def _fit_restricted_ols(design_df, p, fixed_cols=None):
    own_cols = [f"dy_lag_{i}" for i in range(1, p + 1)] + list(fixed_cols or [])
    y = design_df["dy"].astype(float)
    X = sm.add_constant(design_df[own_cols].astype(float), has_constant="add")
    return sm.OLS(y, X).fit()

def _lb_pvalue(resid, lags=8):
    lb = acorr_ljungbox(pd.Series(resid).dropna(), lags=[lags], return_df=True)
    return float(lb["lb_pvalue"].iloc[0])

def _ar_stable(params, p):
    if p == 0:
        return True
    ar = np.array([float(params.get(f"dy_lag_{i}", 0.0)) for i in range(1, p + 1)], dtype=float)
    roots = np.roots(np.r_[1.0, -ar])
    return bool(np.all(np.abs(roots) > 1.0))

def _compare_block_p(full_res, rest_res):
    try:
        return float(full_res.compare_f_test(rest_res)[1])
    except Exception:
        return np.nan

def _rebuild_level(prev_level, pred_dy):
    return float(prev_level * np.exp(pred_dy / 100.0))

def _rolling_oos(design_df, p, q, start_train, fixed_cols=None, show_progress=False, desc="rolling OOS"):
    rows = []
    iterator = range(start_train, len(design_df))
    if show_progress:
        iterator = tqdm(iterator, total=max(0, len(design_df) - start_train), desc=desc, leave=False)

    cols = _feature_cols(p, q, fixed_cols=fixed_cols)

    for origin in iterator:
        train = design_df.iloc[:origin].copy()
        test = design_df.iloc[[origin]].copy()

        y_train = train["dy"].astype(float)
        X_train = sm.add_constant(train[cols].astype(float), has_constant="add")
        X_test = sm.add_constant(test[cols].astype(float), has_constant="add")

        res = sm.OLS(y_train, X_train).fit()

        pred_dy = float(res.predict(X_test).iloc[0])
        actual_dy = float(test["dy"].iloc[0])

        prev_level = float(design_df["y_level"].iloc[origin - 1])
        actual_level = float(test["y_level"].iloc[0])
        pred_level = _rebuild_level(prev_level, pred_dy)

        rows.append({
            "date": test.index[0],
            "actual_dy": actual_dy,
            "pred_dy": pred_dy,
            "actual_level": actual_level,
            "pred_level": pred_level,
            "ae_chg": abs(actual_dy - pred_dy),
            "se_chg": (actual_dy - pred_dy) ** 2,
            "ae_level": abs(actual_level - pred_level),
            "se_level": (actual_level - pred_level) ** 2,
            "ape_level": abs(actual_level - pred_level) / actual_level if actual_level != 0 else np.nan,
        })

    out = pd.DataFrame(rows)
    metrics = {
        "oos_rmse_chg": float(np.sqrt(out["se_chg"].mean())) if len(out) else np.nan,
        "oos_mae_chg": float(out["ae_chg"].mean()) if len(out) else np.nan,
        "oos_rmse_level": float(np.sqrt(out["se_level"].mean())) if len(out) else np.nan,
        "oos_mae_level": float(out["ae_level"].mean()) if len(out) else np.nan,
        "oos_mape_level_pct": float(100.0 * out["ape_level"].mean()) if len(out) else np.nan,
        "n_oos": int(len(out)),
    }
    return out, metrics

def _impulse_response(params, p, q, horizon=26):
    phi = np.array([float(params.get(f"dy_lag_{i}", 0.0)) for i in range(1, p + 1)], dtype=float)
    beta = np.array([float(params.get(f"dx_lag_{j}", 0.0)) for j in range(1, q + 1)], dtype=float)

    irf = np.zeros(horizon + 1)
    for h in range(1, horizon + 1):
        direct = beta[h - 1] if h <= q else 0.0
        feedback = 0.0
        for i in range(1, p + 1):
            if h - i >= 0:
                feedback += phi[i - 1] * irf[h - i]
        irf[h] = direct + feedback

    path = pd.DataFrame({"lag": np.arange(horizon + 1), "response": irf})
    abs_resp = np.abs(path.loc[path["lag"] > 0, "response"].values)
    lags = path.loc[path["lag"] > 0, "lag"].values

    peak_lag = int(lags[np.argmax(abs_resp)]) if abs_resp.sum() > 0 else np.nan
    half_mass_lag = int(lags[np.searchsorted(abs_resp.cumsum(), abs_resp.sum() / 2)]) if abs_resp.sum() > 0 else np.nan
    mean_lag = float(np.sum(lags * abs_resp) / np.sum(abs_resp)) if abs_resp.sum() > 0 else np.nan

    stats = {
        "irf_peak_lag": peak_lag,
        "irf_half_mass_lag": half_mass_lag,
        "irf_mean_lag": mean_lag,
        "recommended_shift_units": int(round(mean_lag)) if pd.notna(mean_lag) else np.nan,
    }
    return path, stats

def screen_benchmark_stage0(
    daily_benchmark_df,
    weekly_refinery_df,
    *,
    daily_date_col,
    weekly_date_col,
    target_col,
    crude_cols,
    product_cols,
    start=None,
    end=None,
    max_p=8,
    max_q=12,
    min_train=156,
    add_month_dummies=False,
    lb_lag=8,
    show_progress=True,
):
    ddf = _to_datetime_index(daily_benchmark_df, daily_date_col)
    wdf = _to_datetime_index(weekly_refinery_df, weekly_date_col)

    candidate_group = {c: "crude" for c in crude_cols}
    candidate_group.update({c: "product" for c in product_cols})
    candidate_cols = [c for c in list(crude_cols) + list(product_cols) if c in ddf.columns]

    wk = pd.DataFrame(index=pd.date_range(ddf.index.min(), ddf.index.max(), freq="W-SAT"))
    for c in candidate_cols:
        wk[c] = _resample_w_sat_mean(ddf[c])

    wk[target_col] = wdf[target_col].astype(float)

    if start is not None:
        wk = wk.loc[pd.to_datetime(start):]
    if end is not None:
        wk = wk.loc[:pd.to_datetime(end)]

    wk = wk[[target_col] + candidate_cols].dropna(how="any").copy()

    fixed_df = _month_dummies(wk.index) if add_month_dummies else pd.DataFrame(index=wk.index)
    fixed_cols = list(fixed_df.columns)

    rows = []
    iterator = candidate_cols
    if show_progress:
        iterator = tqdm(candidate_cols, desc="Stage 0 benchmark screening")

    for cand in iterator:
        pair = _build_design(wk[target_col], wk[cand], p_max=max_p, q_max=max_q, fixed_df=fixed_df)

        for p in range(0, max_p + 1):
            for q in range(1, max_q + 1):
                cols = _feature_cols(p, q, fixed_cols=fixed_cols)
                design = pair[["y_level", "x_level", "dy", "dx"] + cols].copy()
                if len(design) <= (min_train + 10):
                    continue

                train = design.iloc[:min_train].copy()
                full = design.copy()

                try:
                    train_full = _fit_plain_ols(train, p, q, fixed_cols=fixed_cols)
                    train_rest = _fit_restricted_ols(train, p, fixed_cols=fixed_cols)
                    full_full = _fit_plain_ols(full, p, q, fixed_cols=fixed_cols)
                    full_rest = _fit_restricted_ols(full, p, fixed_cols=fixed_cols)
                    _, oos_metrics = _rolling_oos(
                        design,
                        p=p,
                        q=q,
                        start_train=min_train,
                        fixed_cols=fixed_cols,
                        show_progress=False,
                    )

                    row = {
                        "group": candidate_group.get(cand, "unknown"),
                        "candidate": cand,
                        "p": p,
                        "q": q,
                        "bic": float(train_full.bic),
                        "train_lb_p": _lb_pvalue(train_full.resid, lags=lb_lag),
                        "full_lb_p": _lb_pvalue(full_full.resid, lags=lb_lag),
                        "train_block_p": _compare_block_p(train_full, train_rest),
                        "full_block_p": _compare_block_p(full_full, full_rest),
                        "train_stable": _ar_stable(train_full.params, p),
                        "full_stable": _ar_stable(full_full.params, p),
                        **oos_metrics,
                    }
                    row["ok"] = bool(
                        row["train_stable"] and row["full_stable"] and
                        (row["train_lb_p"] >= 0.05) and (row["full_lb_p"] >= 0.05) and
                        (row["train_block_p"] < 0.10) and (row["full_block_p"] < 0.10)
                    )
                    rows.append(row)
                except Exception:
                    continue

    grid = pd.DataFrame(rows)
    if grid.empty:
        raise ValueError("No valid model was estimated in Stage 0.")

    best_each = (
        grid.sort_values(
            ["candidate", "ok", "oos_rmse_level", "full_lb_p", "bic", "q", "p"],
            ascending=[True, False, True, False, True, True, True],
        )
        .groupby("candidate", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    best_each = best_each.sort_values(
        ["ok", "oos_rmse_level", "full_lb_p", "bic"],
        ascending=[False, True, False, True],
    ).reset_index(drop=True)

    winner = best_each.iloc[0].to_dict()
    return {
        "weekly_common_sample": wk,
        "candidate_grid": grid,
        "best_each": best_each,
        "winner": winner,
    }


def _pick_one_existing(df, candidates, label):
    hits = [c for c in candidates if c in df.columns]
    if len(hits) != 1:
        raise ValueError(
            f"{label} 컬럼을 정확히 1개 찾지 못했습니다.\n"
            f"후보={candidates}\n"
            f"실제 발견={hits}\n"
            f"현재 파일 컬럼을 다시 확인하세요."
        )
    return hits[0]


In [ ]:
# =========================
# 3) 전처리 통합 데이터 로딩
# =========================
INTEGRATED_PATH = os.path.join(PROCESSED_PATH, "분석용_일별_통합데이터.csv")

if not os.path.exists(INTEGRATED_PATH):
    raise FileNotFoundError(f"전처리 통합 데이터가 없습니다: {INTEGRATED_PATH}")

integrated_df = pd.read_csv(INTEGRATED_PATH)
integrated_df["date"] = pd.to_datetime(integrated_df["date"])
integrated_df = integrated_df.sort_values("date").copy()

REFINERY_GAS_CANDIDATES = [
    "정유소_세전_보통휘발유",
    "정유사_세전_보통휘발유",
]

REFINERY_DIESEL_CANDIDATES = [
    "정유소_세전_자동차용경유",
    "정유사_세전_자동차용경유",
]

REFINERY_GAS_COL = _pick_one_existing(integrated_df, REFINERY_GAS_CANDIDATES, "휘발유 정유사 세전")
REFINERY_DIESEL_COL = _pick_one_existing(integrated_df, REFINERY_DIESEL_CANDIDATES, "경유 정유사 세전")

required_daily_cols = [
    "date",
    "두바이_원리터",
    "브렌트_원리터",
    "WTI_원리터",
    "휘발유92RON_원리터",
    "경유0.001_원리터",
    "보통휘발유_평균",
    "자동차용경유_평균",
]

missing = [c for c in required_daily_cols if c not in integrated_df.columns]
if missing:
    raise ValueError(f"통합파일에 필요한 컬럼이 없습니다: {missing}")

print("[확인] refinery gasoline col  :", REFINERY_GAS_COL)
print("[확인] refinery diesel col    :", REFINERY_DIESEL_COL)
print("[확인] daily benchmark columns:", [c for c in required_daily_cols if c != "date"])

daily_benchmark_df = integrated_df[[
    "date",
    "두바이_원리터",
    "브렌트_원리터",
    "WTI_원리터",
    "휘발유92RON_원리터",
    "경유0.001_원리터",
]].rename(columns={
    "두바이_원리터": "dubai_krw_l",
    "브렌트_원리터": "brent_krw_l",
    "WTI_원리터": "wti_krw_l",
    "휘발유92RON_원리터": "mogas92_krw_l",
    "경유0.001_원리터": "gasoil_0001_krw_l",
}).copy()

weekly_sparse = integrated_df.set_index("date")[[REFINERY_GAS_COL, REFINERY_DIESEL_COL]].copy()

weekly_refinery_df = (
    weekly_sparse
    .resample("W-SAT")
    .last()
    .rename(columns={
        REFINERY_GAS_COL: "refinery_gasoline_pre_tax",
        REFINERY_DIESEL_COL: "refinery_diesel_pre_tax",
    })
    .dropna(how="all")
    .reset_index()
    .rename(columns={"date": "week_end"})
)

print("\n[weekly_refinery_df sample - rebucketed to W-SAT]")
display(weekly_refinery_df.head(10))

print("\n[week_end day-name check]")
display(weekly_refinery_df["week_end"].dt.day_name().value_counts())

print("\n[n_weekly_obs]")
print(len(weekly_refinery_df))


In [ ]:
# =========================
# 4) benchmark screening 실행 및 저장
# =========================
def ensure_dir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)


def _winner_to_frame(winner: dict) -> pd.DataFrame:
    return pd.DataFrame([winner])


def save_stage0_result(fuel: str, result: dict) -> dict:
    ensure_dir(BENCHMARK_OUTPUT_PATH)

    output_paths = {
        "candidate_grid": os.path.join(BENCHMARK_OUTPUT_PATH, f"{fuel}_stage0_candidate_grid.csv"),
        "best_each": os.path.join(BENCHMARK_OUTPUT_PATH, f"{fuel}_stage0_best_each.csv"),
        "winner": os.path.join(BENCHMARK_OUTPUT_PATH, f"{fuel}_stage0_winner.csv"),
    }

    result["candidate_grid"].to_csv(output_paths["candidate_grid"], index=False, encoding="utf-8-sig")
    result["best_each"].to_csv(output_paths["best_each"], index=False, encoding="utf-8-sig")
    _winner_to_frame(result["winner"]).to_csv(output_paths["winner"], index=False, encoding="utf-8-sig")

    print(f"\n[저장 완료] {fuel}")
    for label, output_path in output_paths.items():
        print(f"- {label}: {output_path}")

    return output_paths


def run_stage0_for_fuel(fuel: str, target_col: str, product_cols: list[str]) -> dict:
    crude_cols = ["dubai_krw_l", "brent_krw_l", "wti_krw_l"]

    result = screen_benchmark_stage0(
        daily_benchmark_df=daily_benchmark_df,
        weekly_refinery_df=weekly_refinery_df,
        daily_date_col="date",
        weekly_date_col="week_end",
        target_col=target_col,
        crude_cols=crude_cols,
        product_cols=product_cols,
        start="2008-05-01",
        end="2025-12-31",
        max_p=8,
        max_q=12,
        min_train=156,
        add_month_dummies=False,
        lb_lag=8,
        show_progress=True,
    )

    best = result["best_each"].copy()
    winner = result["winner"]

    print(f"\n[{fuel} Stage 0 best_each]")
    display(
        best[[
            "group", "candidate", "ok",
            "p", "q",
            "oos_rmse_level", "oos_mae_level", "oos_mape_level_pct",
            "bic", "full_lb_p", "full_block_p"
        ]]
    )

    print(f"\n[{fuel} Stage 0 winner]")
    print(pd.Series(winner)[[
        "group", "candidate", "ok",
        "p", "q", "oos_rmse_level", "bic", "full_lb_p", "full_block_p"
    ]])

    result["output_paths"] = save_stage0_result(fuel, result)
    return result


gasoline_stage0_result = run_stage0_for_fuel(
    fuel="gasoline",
    target_col="refinery_gasoline_pre_tax",
    product_cols=["mogas92_krw_l"],
)

diesel_stage0_result = run_stage0_for_fuel(
    fuel="diesel",
    target_col="refinery_diesel_pre_tax",
    product_cols=["gasoil_0001_krw_l"],
)

selected_benchmarks = pd.DataFrame([
    {"fuel": "gasoline", **gasoline_stage0_result["winner"]},
    {"fuel": "diesel", **diesel_stage0_result["winner"]},
])

selected_benchmarks_path = os.path.join(BENCHMARK_OUTPUT_PATH, "stage0_selected_benchmarks.csv")
selected_benchmarks.to_csv(selected_benchmarks_path, index=False, encoding="utf-8-sig")

print("\n[선택 benchmark 요약]")
display(selected_benchmarks[["fuel", "group", "candidate", "ok", "p", "q", "oos_rmse_level", "bic", "full_lb_p", "full_block_p"]])
print(f"[저장 완료] selected_benchmarks: {selected_benchmarks_path}")
